In [0]:
%pip install /Workspace/Users/gpuchalski@kumc.edu/PFTSleep/
dbutils.library.restartPython()

In [0]:
dbutils.library.restartPython()


In [0]:
# -------------------------
# 1) Imports
# -------------------------
from pathlib import Path
import os
import time
import numpy as np
import torch
import json

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from pftsleep.transformers import PatchTFTSimple
from pftsleep.slumber import (
    SelfSupervisedTimeFrequencyDataset,
    ALL_FREQUENCY_FILTERS,
    VOLTAGE_CHANNELS
)




In [0]:
# -------------------------
# 2) User-configurable variables
# -------------------------
# ZARR FILES
zarr_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")
num_files = 1229


# SIGNAL / WINDOWING
frequency = 125
win_length = 750
hop_length = 750
max_seq_len_sec = 8 * 3600
window_size_sec = win_length / frequency

# PERFORMANCE
workers = 16
device = "cuda" if torch.cuda.is_available() else "cpu"

# ENCODERS / CHECKPOINTS (add more here as you expand)
pft_ckpt_path = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/models/pft_sleep_encoder.ckpt")

# OUTPUT DIRECTORIES
save_PCA_plot_here = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/pca_plots")
save_UMAP_plot_here = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/umap_plots")
cache_dir = Path("/Volumes/kumc_sleep/sleep_studies/shhs_data/pftsleep_cache/")

save_PCA_plot_here.mkdir(parents=True, exist_ok=True)
save_UMAP_plot_here.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

# PCA
given_n_components = 512
pca_target_var = 0.95  # "PCA95% variance"

# UMAP (visualization only)
umap_n_neighbors = 15
umap_min_dist = 0.1
umap_n_components = 3
umap_metric = "euclidean"
umap_random_state = 42

# K-selection sweep
k_values = list(range(8, 14))  # adjust as needed
seeds = [0, 1, 2, 3, 4]

# LSH parameters (raw-space comparison)
lsh_n_bits = 16
lsh_top_buckets = 2000



In [0]:
# -------------------------
# 3) Channel definitions
# -------------------------
channels = [
    ["ECG", "ECG (L-R)", "EKG"],
    ["EOG(L)", "E1", "E1-M1", "EOG-L"],
    ["EMG", "cchin_1", "chin", "EMG (L-R)"],
    ["EEG", "C3-M2", "C4-M1", "C3-M2", "EEG3"],
    ["SaO2", "spo2", "SpO2"],
    ["THOR RES", "thorax", "Thoracic", "Chest", "Thor"],
    ["ABDO RES", "abdomen", "Abdominal", "ABD", "Abdo"],
]
c_in = len(channels)


In [0]:
# -------------------------
# 4) Load Zarrs
# -------------------------
zarr_files = sorted([p for p in zarr_dir.glob("*.zarr")])[:num_files]
print(f"Found {len(zarr_files)} zarr files (using num_files={num_files}).")
assert len(zarr_files) > 0, "No Zarr files found."

def extract_zarr_uid(path: Path) -> str:
    return path.stem

zarr_files = sorted([p for p in zarr_dir.glob("*.zarr")])[:num_files]
 
# Extract UIDs
zarr_uids = [extract_zarr_uid(p) for p in zarr_files]
 
# Sort to guarantee deterministic ordering
unique_sorted_uids = sorted(zarr_uids)
 
# UID → numeric ID
uid_to_numeric = {uid: i for i, uid in enumerate(unique_sorted_uids)}
 
# File path (string) → numeric ID
file_to_numeric_id = {
    str(p): uid_to_numeric[p.stem]
    for p in zarr_files
}

import json
with open(cache_dir / "zarr_id_map.json", "w") as f:
    json.dump(uid_to_numeric, f, indent=2)


In [0]:

# -------------------------
# 5) Dataset / Loader
# -------------------------
max_seq_len = max_seq_len_sec * frequency

dataset = SelfSupervisedTimeFrequencyDataset(
    zarr_files=zarr_files,
    channels=channels,
    frequency=frequency,
    trim_wake_epochs=True,
    return_hypnogram_every_sec=30,
    hypnogram_frequency=1,
    hypnogram_padding_mask=-100,
    scale_channels=False,
    start_offset_sec=0,
    clip_interpolations=None,
    include_partial_samples=True,
    return_sequence_padding_mask=True,
    butterworth_filters=ALL_FREQUENCY_FILTERS,
    median_filter_kernel_size=3,
    voltage_channels=VOLTAGE_CHANNELS,
    max_seq_len_sec=max_seq_len_sec,
    sample_seq_len_sec=max_seq_len_sec,
    sample_stride_sec=max_seq_len_sec,
)

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=workers,
    persistent_workers=True,
    pin_memory=False,
)

print("Dataset + loader ready.")


In [0]:
# -------------------------
# 6) Encoder registry (extendable)
# -------------------------
def build_pftsleep_encoder() -> torch.nn.Module:
    model = PatchTFTSimple(
        c_in=c_in,
        win_length=win_length,
        hop_length=hop_length,
        max_seq_len=max_seq_len,
        use_revin=True,
        dim1reduce=False,
        affine=True,
        use_flash_attn=False,
        augmentations=["jitter_zero_mask"],
        mask_ratio=0.1,
        n_layers=3,
        d_model=512,
        n_heads=4,
        shared_embedding=False,
        d_ff=2048,
        norm="BatchNorm",
        attn_dropout=0.0,
        dropout=0.1,
        act="gelu",
        res_attention=True,
        pre_norm=False,
        store_attn=False,
        pretrain_head=False,
    )
    return model

def load_pftsleep_weights(model: torch.nn.Module, ckpt_path: Path) -> torch.nn.Module:
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state_dict = {
        k.replace("model.", ""): v
        for k, v in ckpt["state_dict"].items()
        if k.startswith("model.")
    }
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    
    # Enable gradient checkpointing to reduce memory usage
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
    
    return model

# Placeholder hooks for other encoders (SleepFM, etc.)
# Implement build/load functions similarly, then register below.
def build_sleepfm_encoder():
    raise NotImplementedError("Add SleepFM encoder construction here.")

def load_sleepfm_weights(model, ckpt_path: Path):
    raise NotImplementedError("Add SleepFM checkpoint loading here.")

ENCODER_REGISTRY = {
    "PFTSleep": {
        "build": build_pftsleep_encoder,
        "load": lambda m: load_pftsleep_weights(m, pft_ckpt_path),
    },
    # "SleepFM": {"build": build_sleepfm_encoder, "load": lambda m: load_sleepfm_weights(m, sleepfm_ckpt_path)},
}


In [0]:
# ------------------------------------------------------------

# 7) Extract + cache latents per encoder (SHARD MODE, FIXED)

# ------------------------------------------------------------
 
from pathlib import Path

import numpy as np

import torch

from tqdm import tqdm
 
def cache_tag(

    encoder_name: str,

    num_files: int,

    frequency: int,

    win_length: int,

    hop_length: int,

    max_seq_len_sec: int,

) -> str:

    return (

        f"{encoder_name}"

        f"__files{num_files}"

        f"__freq{frequency}"

        f"__win{win_length}"

        f"__hop{hop_length}"

        f"__max{max_seq_len_sec}"

    )
 
 
def extract_latents_cached(

    encoder_name: str,

    encoder: torch.nn.Module,

    loader,

    cache_dir: Path,

    window_size_sec: float,

    device: str = "cpu",

):
 
    tag = cache_tag(

        encoder_name,

        num_files,

        frequency,

        win_length,

        hop_length,

        max_seq_len_sec,

    )
 
    dbfs_cache_dir = Path("/dbfs/tmp/pftsleep_cache")

    dbfs_cache_dir.mkdir(parents=True, exist_ok=True)
 
    shard_dir = dbfs_cache_dir / f"{tag}_shards"

    shard_dir.mkdir(parents=True, exist_ok=True)
 
    night_id_path = dbfs_cache_dir / f"night_id__{tag}.npy"

    time_idx_path = dbfs_cache_dir / f"time_idx__{tag}.npy"

    zarr_file_idx_path = dbfs_cache_dir / f"zarr_file_idx__{tag}.npy"
 
    # If metadata already exists, assume shards exist

    if night_id_path.exists() and time_idx_path.exists() and zarr_file_idx_path.exists():

        print(f"[{encoder_name}] Loading cached metadata...")

        night_id = np.load(night_id_path)

        time_idx = np.load(time_idx_path)

        zarr_file_idx = np.load(zarr_file_idx_path)

        return shard_dir, night_id, time_idx, zarr_file_idx
 
    print(f"[{encoder_name}] Extracting latents (6-second windows)...")
 
    encoder = encoder.to(device)

    encoder.eval()
 
    night_ids = []

    time_idxs = []

    zarr_file_idxs = []
 
    with torch.no_grad():

        for batch_idx, batch in enumerate(

            tqdm(loader, desc=f"Encoding ({encoder_name})", unit="file")

        ):
 
            x = batch[0].to(device)

            sequence_padding_mask = batch[2].to(device)
 
            z_latent = encoder(x, sequence_padding_mask=sequence_padding_mask)

            z_latent = z_latent.squeeze(0)
 
            # ------------------------------------------------------------

            # FIX: one embedding per 6-second window

            # ------------------------------------------------------------

            if z_latent.ndim == 3:

                # Expected problematic shape: [7, 512, T]

                if z_latent.shape[0] == 7 and z_latent.shape[1] == 512:

                    z_latent = z_latent.mean(dim=0)  # -> [512, T]

                    z_latent = z_latent.T            # -> [T, 512]

                else:

                    raise ValueError(f"Unexpected z_latent shape: {tuple(z_latent.shape)}")
 
            elif z_latent.ndim == 2:

                if z_latent.shape[1] != 512:

                    raise ValueError(f"Unexpected z_latent shape: {tuple(z_latent.shape)}")

            else:

                raise ValueError(f"Unexpected z_latent.ndim={z_latent.ndim}")
 
            # Sanity check (should be ~4800 per 8-hour file)

            expected_windows = int((8 * 3600) / window_size_sec)

            if batch_idx == 0:

                print(f"First file windows: {z_latent.shape[0]} (expected ~{expected_windows})")
 
            z_np = z_latent.detach().cpu().numpy().astype(np.float16)
 
            # Save shard

            np.save(shard_dir / f"Z_part_{batch_idx:05d}.npy", z_np)
 
            n_windows = z_np.shape[0]
 
            night_ids.append(np.full(n_windows, batch_idx, dtype=np.int32))

            time_idxs.append(np.arange(n_windows, dtype=np.float32) * window_size_sec)

            zarr_file_idxs.append(np.full(n_windows, batch_idx, dtype=np.int32))
 
    night_id = np.concatenate(night_ids)

    time_idx = np.concatenate(time_idxs)

    zarr_file_idx = np.concatenate(zarr_file_idxs)
 
    np.save(night_id_path, night_id)

    np.save(time_idx_path, time_idx)

    np.save(zarr_file_idx_path, zarr_file_idx)
 
    print(f"[{encoder_name}] Extraction complete.")

    print(f"Total windows: {len(night_id):,}")
 
    return shard_dir, night_id, time_idx, zarr_file_idx
 
 
# ------------------------------------------------------------

# Build + extract

# ------------------------------------------------------------
 
LATENTS = {}
 
for enc_name, spec in ENCODER_REGISTRY.items():

    model = spec["build"]()

    model = spec["load"](model)
 
    shard_dir, night_id, time_idx, zarr_file_idx = extract_latents_cached(

        encoder_name=enc_name,

        encoder=model,

        loader=loader,

        cache_dir=cache_dir,

        window_size_sec=window_size_sec,

        device=device,

    )
 
    LATENTS[enc_name] = {

        "shard_dir": shard_dir,

        "night_id": night_id,

        "time_idx": time_idx,

        "zarr_file_idx": zarr_file_idx,

        "zarr_files": [str(p) for p in zarr_files],

    }
 
# Use PFTSleep latents

shard_dir = LATENTS["PFTSleep"]["shard_dir"]

night_id = LATENTS["PFTSleep"]["night_id"]

time_idx = LATENTS["PFTSleep"]["time_idx"]

zarr_file_idx = LATENTS["PFTSleep"]["zarr_file_idx"]
 
print("Metadata shapes:")

print("night_id:", night_id.shape)

print("time_idx:", time_idx.shape)

print("zarr_file_idx:", zarr_file_idx.shape)
 

# Representation Extraction Complete ✅

## Saved Files (for Clustering_Business notebook)

This notebook extracts latent representations from sleep study zarr files and saves the following metadata:

### Cache Files Location
`cache_dir`: Configured in Cell 3

### Files Saved (per encoder)
Pattern: `{variable}__{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}.npy`

1. **Z__{tag}.npy** - Latent representations [N_windows, 512]
2. **night_id__{tag}.npy** - Night/batch identifier for each window
3. **time_idx__{tag}.npy** - Time index in seconds for each window
4. **zarr_file_idx__{tag}.npy** - Numeric zarr file ID for each window (maps to zarr_id_map.json)
5. **windows_start_idx__{tag}.npy** - Start index in original zarr for each window
6. **zarr_files_list__{tag}.npy** - List of zarr file paths (for mapping back to source)
7. **zarr_id_map.json** - Mapping from zarr UID to numeric ID

## Next Steps

**Run the Clustering_Business notebook** which will:
1. Load the cached latent representations
2. Perform LSH-means initialization (GPU-accelerated)
3. Run GPU KMeans clustering
4. Assign cluster IDs to each data point
5. Map cluster IDs back to source zarr files using the metadata

## GPU Acceleration

✅ **Enabled**: This workflow uses GPU-accelerated KMeans from cuML
- Faster clustering on large datasets
- Runs entirely on GPU (T4 in your case)
- Compatible with the LSH-means initialization strategy